# Minh họa quy trình tiền xử lý dữ liệu

Notebook này minh họa các bước tiền xử lý dữ liệu cho bài toán phân loại ngôn ngữ ký hiệu Việt Nam:
1. **Xử lý dữ liệu trùng lặp** - Loại bỏ class AAA
2. **Resize ảnh** - Chuẩn hóa kích thước về 224×224
3. **Data Augmentation** - Flip, brightness, contrast
4. **Normalization** - ResNet50 preprocessing

Các hình ảnh được tạo ra phù hợp để đưa vào slide thuyết trình.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
import os
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

## 1. Load dữ liệu gốc

In [ ]:
# Giải nén dữ liệu gốc
!unzip -o -q "/content/drive/MyDrive/NTTU/HK2-2026/AI_2/Cuối kỳ/data/raw/archive.zip" -d /content/data

# Đường dẫn dữ liệu gốc
raw_data_dir = '/content/data/Dataset'
print("Đã giải nén dữ liệu gốc")

In [ ]:
# Lấy danh sách các class (sau khi đã loại bỏ AAA)
classes = sorted([d for d in os.listdir(raw_data_dir) if os.path.isdir(os.path.join(raw_data_dir, d))])
# Loại bỏ AAA nếu tồn tại
if 'AAA' in classes:
    classes.remove('AAA')
print(f"Số lượng class: {len(classes)}")
print(f"Danh sách class: {classes}")

## 2. Hàm tiện ích

In [ ]:
def load_raw_image(class_name, image_index=0):
    """Load một ảnh gốc từ thư mục raw data"""
    class_path = os.path.join(raw_data_dir, class_name)
    image_files = sorted([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    if image_index >= len(image_files):
        image_index = 0
    
    image_path = os.path.join(class_path, image_files[image_index])
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    return image.numpy(), image_files[image_index]

def apply_augmentation(image):
    """Áp dụng data augmentation"""
    # Lật ngang ngẫu nhiên (cố định seed để demo)
    image = tf.image.flip_left_right(image)
    
    # Thay đổi độ sáng
    image = tf.image.adjust_brightness(image, delta=0.15)
    
    # Thay đổi độ tương phản
    image = tf.image.adjust_contrast(image, contrast_factor=1.1)
    
    # Đảm bảo giá trị trong khoảng [0, 255]
    image = tf.clip_by_value(image, 0, 255)
    
    return image.numpy().astype('uint8')

def apply_resnet_preprocessing(image):
    """Áp dụng ResNet50 preprocessing"""
    # Chuyển sang float32
    image_float = tf.cast(image, tf.float32)
    
    # Áp dụng ResNet50 preprocessing
    image_preprocessed = preprocess_input(image_float)
    
    # Chuyển về dạng có thể hiển thị (denormalize để xem)
    # ResNet preprocessing: RGB -> BGR và trừ mean
    # Để hiển thị, ta cần reverse lại
    image_display = image_preprocessed.numpy().copy()
    image_display[:, :, 0] += 103.939  # Blue
    image_display[:, :, 1] += 116.779  # Green
    image_display[:, :, 2] += 123.68   # Red
    # BGR -> RGB
    image_display = image_display[:, :, ::-1]
    image_display = np.clip(image_display, 0, 255).astype('uint8')
    
    return image_display

## 3. Minh họa quy trình tiền xử lý đầy đủ

In [ ]:
def visualize_full_preprocessing_pipeline(class_name='A', image_index=0, save_figure=True):
    """
    Minh họa toàn bộ quy trình tiền xử lý:
    1. Ảnh gốc
    2. Resize 224x224
    3. Data Augmentation
    4. Normalization (ResNet50)
    """
    # Bước 1: Load ảnh gốc
    raw_image, filename = load_raw_image(class_name, image_index)
    
    # Bước 2: Resize
    resized_image = tf.image.resize(raw_image, [224, 224]).numpy().astype('uint8')
    
    # Bước 3: Data Augmentation
    augmented_image = apply_augmentation(resized_image)
    
    # Bước 4: Normalization (ResNet50 preprocessing)
    normalized_image = apply_resnet_preprocessing(augmented_image)
    
    # Tạo figure với 4 cột
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    # Hiển thị từng bước
    axes[0].imshow(raw_image)
    axes[0].set_title(f'Bước 1: Ảnh gốc\nClass: {class_name}\nSize: {raw_image.shape[:2]}', 
                     fontsize=11, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(resized_image)
    axes[1].set_title(f'Bước 2: Resize\nSize: {resized_image.shape[:2]}\nChuẩn hóa kích thước', 
                     fontsize=11, fontweight='bold')
    axes[1].axis('off')
    
    axes[2].imshow(augmented_image)
    axes[2].set_title(f'Bước 3: Augmentation\nFlip + Brightness + Contrast\nTăng tính đa dạng', 
                     fontsize=11, fontweight='bold')
    axes[2].axis('off')
    
    axes[3].imshow(normalized_image)
    axes[3].set_title(f'Bước 4: Normalization\nResNet50 Preprocessing\nSẵn sàng training', 
                     fontsize=11, fontweight='bold')
    axes[3].axis('off')
    
    plt.suptitle('QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_figure:
        output_path = '/content/drive/MyDrive/NTTU/HK2-2026/AI_2/Cuối kỳ/reports/figures/preprocessing_pipeline.png'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"✓ Đã lưu hình ảnh tại: {output_path}")
    
    plt.show()

In [ ]:
# Hiển thị quy trình đầy đủ
visualize_full_preprocessing_pipeline(class_name='A', image_index=5, save_figure=True)

## 4. So sánh trước và sau tiền xử lý (nhiều mẫu)

In [ ]:
def compare_before_after(selected_classes=['A', 'B', 'C', 'D', 'E', 'G'], save_figure=True):
    """
    So sánh ảnh trước (gốc) và sau (đã xử lý đầy đủ) cho nhiều class
    """
    num_classes = len(selected_classes)
    fig, axes = plt.subplots(2, num_classes, figsize=(num_classes * 3, 6))
    
    for i, class_name in enumerate(selected_classes):
        # Load ảnh gốc
        raw_image, filename = load_raw_image(class_name, 0)
        
        # Áp dụng toàn bộ quy trình tiền xử lý
        resized = tf.image.resize(raw_image, [224, 224]).numpy().astype('uint8')
        augmented = apply_augmentation(resized)
        processed = augmented  # Hiển thị sau augmentation (trước normalization để dễ nhìn)
        
        # Hiển thị ảnh gốc (hàng trên)
        axes[0, i].imshow(raw_image)
        axes[0, i].set_title(f'Trước\nClass: {class_name}\n{raw_image.shape[:2]}', 
                            fontsize=10, fontweight='bold')
        axes[0, i].axis('off')
        
        # Hiển thị ảnh đã xử lý (hàng dưới)
        axes[1, i].imshow(processed)
        axes[1, i].set_title(f'Sau\nClass: {class_name}\n{processed.shape[:2]}', 
                            fontsize=10, fontweight='bold')
        axes[1, i].axis('off')
    
    plt.suptitle('SO SÁNH TRƯỚC VÀ SAU TIỀN XỬ LÝ', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    if save_figure:
        output_path = '/content/drive/MyDrive/NTTU/HK2-2026/AI_2/Cuối kỳ/reports/figures/preprocessing_comparison.png'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"✓ Đã lưu hình ảnh tại: {output_path}")
    
    plt.show()

In [ ]:
# So sánh 6 class
compare_before_after(selected_classes=['A', 'B', 'C', 'D', 'E', 'G'], save_figure=True)

## 5. Minh họa Data Augmentation

In [ ]:
def visualize_augmentation_techniques(class_name='A', image_index=0, save_figure=True):
    """
    Minh họa từng kỹ thuật augmentation riêng biệt
    """
    # Load và resize ảnh
    raw_image, filename = load_raw_image(class_name, image_index)
    base_image = tf.image.resize(raw_image, [224, 224]).numpy().astype('uint8')
    
    # Áp dụng từng kỹ thuật riêng biệt
    flipped = tf.image.flip_left_right(base_image).numpy().astype('uint8')
    
    brightness_up = tf.image.adjust_brightness(base_image, delta=0.2)
    brightness_up = tf.clip_by_value(brightness_up, 0, 255).numpy().astype('uint8')
    
    contrast_up = tf.image.adjust_contrast(base_image, contrast_factor=1.3)
    contrast_up = tf.clip_by_value(contrast_up, 0, 255).numpy().astype('uint8')
    
    # Kết hợp tất cả
    combined = apply_augmentation(base_image)
    
    # Tạo figure
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    # Hàng 1
    axes[0, 0].imshow(base_image)
    axes[0, 0].set_title('Ảnh gốc (sau resize)', fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(flipped)
    axes[0, 1].set_title('Random Flip Left-Right\n(50% xác suất)', fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(brightness_up)
    axes[0, 2].set_title('Random Brightness\n(±20%)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Hàng 2
    axes[1, 0].imshow(contrast_up)
    axes[1, 0].set_title('Random Contrast\n(80%-120%)', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(combined)
    axes[1, 1].set_title('Kết hợp tất cả\n(Flip + Brightness + Contrast)', fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Ô trống - thêm text mô tả
    axes[1, 2].text(0.5, 0.5, 
                   'Mục đích:\n\n'
                   '• Tăng tính đa dạng\n'
                   '• Giảm overfitting\n'
                   '• Mô phỏng điều kiện\n'
                   '  thực tế\n\n'
                   'Lưu ý:\n'
                   '• Chỉ áp dụng cho\n'
                   '  tập Train\n'
                   '• Không dùng rotation',
                   ha='center', va='center', fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    axes[1, 2].axis('off')
    
    plt.suptitle(f'CÁC KỸ THUẬT DATA AUGMENTATION - Class: {class_name}', 
                fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    if save_figure:
        output_path = '/content/drive/MyDrive/NTTU/HK2-2026/AI_2/Cuối kỳ/reports/figures/augmentation_techniques.png'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"✓ Đã lưu hình ảnh tại: {output_path}")
    
    plt.show()

In [ ]:
# Hiển thị các kỹ thuật augmentation
visualize_augmentation_techniques(class_name='A', image_index=3, save_figure=True)

## 6. Thống kê tổng quan

In [ ]:
def print_preprocessing_summary():
    """
    In ra thống kê tổng quan về quy trình tiền xử lý
    """
    print("="*70)
    print("TỔNG QUAN QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU")
    print("="*70)
    
    print("\n📊 THÔNG TIN DỮ LIỆU:")
    print("   • Số lớp ban đầu: 26 lớp")
    print("   • Số lớp sau xử lý: 25 lớp (loại bỏ AAA)")
    print("   • Tổng số mẫu: 25,000 ảnh")
    print("   • Phân chia: Train 80% | Val 10% | Test 10%")
    
    print("\n🔧 CÁC BƯỚC TIỀN XỬ LÝ:")
    print("   1. Xử lý dữ liệu trùng lặp")
    print("      → Loại bỏ class AAA (trùng với AA)")
    print("\n   2. Resize ảnh")
    print("      → Chuẩn hóa kích thước: 224×224×3")
    print("      → Phù hợp với input của ResNet50")
    print("\n   3. Data Augmentation (chỉ Train set)")
    print("      → Random Flip Left-Right (50%)")
    print("      → Random Brightness (±20%)")
    print("      → Random Contrast (80%-120%)")
    print("\n   4. Normalization")
    print("      → ResNet50 preprocessing")
    print("      → RGB → BGR + subtract ImageNet mean")
    
    print("\n💾 LƯU TRỮ DỮ LIỆU:")
    print("   • Định dạng: TFRecord + GZIP compression")
    print("   • Dung lượng gốc: ~14.36 GB")
    print("   • Dung lượng sau nén: ~313 MB")
    print("   • Tỷ lệ nén: 45.8 lần")
    
    print("\n✅ LỢI ÍCH:")
    print("   ✓ Chuẩn hóa kích thước đầu vào")
    print("   ✓ Tăng tính đa dạng của dữ liệu")
    print("   ✓ Giảm overfitting")
    print("   ✓ Tiết kiệm dung lượng lưu trữ")
    print("   ✓ Tăng tốc độ training")
    
    print("\n" + "="*70)

In [ ]:
# In thống kê
print_preprocessing_summary()

## 7. Tạo tất cả hình ảnh cho slide

In [ ]:
# Tạo tất cả hình ảnh cần thiết cho slide
print("Đang tạo các hình ảnh cho slide...\n")

print("1. Quy trình tiền xử lý đầy đủ:")
visualize_full_preprocessing_pipeline(class_name='A', image_index=5, save_figure=True)

print("\n2. So sánh trước và sau:")
compare_before_after(selected_classes=['A', 'B', 'C', 'D', 'E', 'G'], save_figure=True)

print("\n3. Các kỹ thuật augmentation:")
visualize_augmentation_techniques(class_name='A', image_index=3, save_figure=True)

print("\n" + "="*70)
print("✓ Hoàn tất! Tất cả hình ảnh đã được lưu vào:")
print("  /content/drive/MyDrive/NTTU/HK2-2026/AI_2/Cuối kỳ/reports/figures/")
print("="*70)